# [Semantic Kernel 05 - Multiple Agents](https://devblogs.microsoft.com/semantic-kernel/introducing-agents-in-semantic-kernel/)
Up until today, we've demonstrated how you could use components of Semantic Kernel to build agents. With just a few lines of code, you can use a chat completion model to answer user’s questions and to automatically invoke plugins as necessary.<br/>
What was missing, however, was a first-class agent abstraction. Not only would this simplify code by consolidating logic, but it would also ensure that there is a common contract on how to interact with an agent. This may seem like a small feat, but it has allowed us to build a multi-agent framework that allows agents to coordinate with one another while also simplifying the code you need to write.<br/>
With our latest Python (1.6.0) and .NET releases (1.18.0 RC1), Semantic Kernel now provides a first-class abstraction for agents. This reduces much of the complexity required to build a standard chat experience while also providing a standardized API to interact with them. With this release, we’re providing two out-of-the-box agents: Assistant API agents and Chat completion agents.

# Constants and Libraries

In [1]:
import os, json, sys, random
from dotenv import load_dotenv # requires python-dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior # Auto(), Required() or NoneInvoke()
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents.chat_message_content import ChatMessageContent

from IPython.display import Markdown, display # nothing to pip install

load_dotenv("./../config/credentials_my.env")
chatcompletion_service_id = "chatcompletion_service_id"
maximum_iterations = 50

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Define the Kernel

In [2]:
kernel = Kernel()
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001ABEE76DAC0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Create an AzureChatCompletion AI Service and add it to the Kernel

In [3]:
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='chatcompletion_service_id', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001ABFFD61310>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001ABEE76DAC0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [4]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)

# Create the agents
The agent `service_id` specified in `ChatCompletionAgent` must match one of the services defined in `Kernel.services`

In [15]:
user_proxy = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="user_proxy",
    description="Orchestrates the multiple personas involved in a discussion",
    kernel=kernel,    
    instructions="""
        Tu sei incaricato di iniziare e moderare la conversazione.
        Tu non hai alcuna conoscenza tecnica. Non dare **MAI** suggerimenti tecnici, devi limitarti a coordinare la conversazione.
        Il tuo ruolo è di fornire una piattaforma dove le persone possono interagire, mostrando le loro inclinazioni e abilità.
        Tuo incarico: facilita una conversazione discorsiva e coinvolgente fra Mauro, Aleksa, Gabriel e Federica senza imporre alcun suggerimento.
        Non permettere agli interlocutori di esprimere commenti su aree di tecnologie diverse da quella su cui loro sono esperti.
    """
)

In [16]:
mauro = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="mauro",
    description="A Microsoft Cloud Solution Architect expert just at Data Engineering and nothing more",
    kernel=kernel,    
    instructions="""
        Tu sei Mauro. Mauro è un Cloud Solution Architect specializzato in Data Engineering, 
        esperto nella progettazione e implementazione di soluzioni per la gestione e l'analisi di grandi volumi di dati, 
        con un forte impegno verso l'ottimizzazione delle prestazioni e l'integrità dei dati.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """
)

In [17]:
aleksa = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="aleksa",
    description="A Microsoft Cloud Solution Architect expert just at Data Science and Generative AI and nothing more",
    kernel=kernel,    
    instructions="""
        Tu sei Aleksa. Sei un Cloud Solution Architect specializzata in Data Science, con una profonda conoscenza 
        delle tecniche di machine learning, intelligenza artificiale e generative AI, appassionata di trasformare dati complessi 
        in insights utili per guidare le decisioni aziendali.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """
)

In [18]:
gabriel = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="gabriel",
    description="A Microsoft Cloud Solution Architect expert just at Security and nothing more",
    kernel=kernel,    
    instructions="""
        Tu sei Gabriel. Sei Cloud Solution Architect esperto di sicurezza, dedicato a progettare e implementare 
        soluzioni di sicurezza cloud robuste e scalabili, con un forte impegno verso la protezione dei dati e la conformità alle normative.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """
)

In [19]:
federica = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="federica",
    description="A Microsoft Cloud Solution Architect expert just in Development Tools and nothing more",
    kernel=kernel,    
    instructions="""
        Tu sei Federica. Sei Cloud Solution Architect specializzata in tecnologie di sviluppo, con una vasta esperienza 
        nella creazione di applicazioni cloud-native e microservizi, sempre alla ricerca di innovazioni che migliorino 
        l'efficienza e la scalabilità delle soluzioni software.        
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """
)

# Termination Strategy definition

In [20]:
class ApprovalTerminationStrategy(TerminationStrategy):
    """A strategy for determining when an agent should terminate."""

    async def should_agent_terminate(self, agent, history):
        """Check if the agent should terminate."""
        return "approved" in history[-1].content.lower()

# Create Group Chat

In [21]:
chat = AgentGroupChat(
    agents = [user_proxy, mauro, aleksa, gabriel, federica],
    termination_strategy=ApprovalTerminationStrategy(agents=[user_proxy], maximum_iterations=maximum_iterations),
)

# Create a user message and add it to a blank history

In [22]:
input = "Your objective is to create the agenda for a 2 hours presentation on the most recent Azure technologies."

await chat.add_chat_message(ChatMessageContent(role=AuthorRole.USER, content=input))
print(f"# {AuthorRole.USER}: '{input}'")

# AuthorRole.USER: 'Your objective is to create the agenda for a 2 hours presentation on the most recent Azure technologies.'


In [23]:
async for content in chat.invoke():
    print(f"# {content.role} - {content.name or '*'}: '{content.content}'")

# AuthorRole.ASSISTANT - user_proxy: 'Mi dispiace, non posso aiutarti con la creazione dell'agenda per una presentazione sulle tecnologie Azure. Tuttavia, posso assisterti facilitando una conversazione tra Mauro, Aleksa, Gabriel e Federica. Sentitevi liberi di iniziare o di condividere il vostro contesto e le vostre aspettative!'
# AuthorRole.ASSISTANT - mauro: 'Sicuro, posso aiutarti. Ecco una proposta:

1. Introduzione (10 min)
2. Azure Data Factory (20 min)
3. Azure Synapse Analytics (20 min)
4. Azure Databricks (20 min)
5. Power BI su Azure (15 min)
6. Azure Machine Learning (15 min)
7. Best Practices in Azure (10 min)
8. Domande e Risposte (10 min)

Ti sembra un buon punto di partenza?'
# AuthorRole.ASSISTANT - aleksa: 'Sì, è un ottimo punto di partenza. Potremmo forse inserire una breve demo pratica da qualche parte?'
# AuthorRole.ASSISTANT - gabriel: 'Ottima idea, una demo pratica potrebbe rendere la presentazione più coinvolgente. Potremmo inserire una demo durante la parte su 

In [14]:
print(f"# IS COMPLETE: {chat.is_complete}")

# IS COMPLETE: False
